## **Research Project Section: Data Pre-processing and Management**

### **Overview**
In this phase, the primary focus is to prepare and organize the data by reading and processing large audio recordings (WAV files) and their corresponding annotation files. The process involves extracting relevant spectrogram fragments (representing calls) and storing them in a structured format for further analysis.

### **1. General Data Processing Workflow**

#### **Data Collection**
- Each `*.wav` audio file is paired with its corresponding annotation file `*.Table.1.selections.txt`.
- The annotation file provides detailed information about the number of calls (rows in the file) and their respective time and frequency ranges.

#### **Spectrogram Generation**
- A single large spectrogram is generated for the entire audio file using fixed parameters (e.g., `signal.spectrogram`). The following configurations are applied:
  - **Sampling rate:** Resample audio files to a common rate (e.g., 16 kHz) if necessary.
  - **Spectrogram parameters:** `nperseg`, `nfft`, `noverlap`, and `window`.
- Spectrogram fragments corresponding to each annotated call are extracted.
- Frequency and time constraints (e.g., 20 Hz to 1000 Hz) are optionally applied based on the annotation data.

### **2. Size Standardization**
- A comprehensive analysis of all annotations is performed to identify the maximum call duration and frequency range (i.e., "the longest and widest call").
- To ensure uniformity, zero padding or other alignment techniques (e.g., time stretching) are applied to shorter spectrograms.
- **Objective:** Ensure that all spectrogram fragments have a standardized shape, such as `(F × T)`.

### **3. Creation of "No-call" Fragments**
- Spectrogram fragments representing "no-call" segments are generated by selecting random time intervals that do not overlap with any calls.
- These "no-call" fragments are sized identically to the call spectrograms `(F × T)` and maintain the same frequency range.

### **4. Data Storage**

#### **Saving Processed Data**
- Each extracted spectrogram (call or "no-call") is saved as a 2D array in NumPy format (`.npy` or `.npz`).
- Metadata files, saved in `.csv` format or as Python structures, accompany the spectrogram data and include:
  - **Call type** (e.g., Rupe A, B, Moan, etc.).
  - **Start and end time** within the original audio file.
  - **Original file ID** and any other necessary fields.

### **5. Directory Structure**
The processed data is organized in the following directory structure:
```
**outcome/**
├── processed_calls/ │ 
├── call_001.npy │ 
├── call_002.npy │ 
└── ... ├── processed_no_calls/ │ 
├── no_call_001.npy │ 
├── no_call_002.npy │ 
└── ... 
└── metadata.csv
```
### **6. Additional Considerations**
- Ensure that sample rates and spectrogram dimensions are consistent across all files to avoid mismatches during analysis.
- Implement verification checks after saving the data to ensure file integrity and data accuracy.

By following this structured approach to data pre-processing, the dataset is prepared for efficient and reliable downstream analysis, facilitating accurate research outcomes.

In [17]:
# Import libs
import os
import matplotlib.pyplot as plt
from scipy import signal
from scipy.io import wavfile
import numpy as np
import pandas as pd
from matplotlib.colors import LogNorm
import glob

In [18]:
# Define folders path
'''
Folders name list:
    Rupes A and B
    Moan
    Guttural rupe
    Grey Seal Data Additional
'''

DATA_ROOT = 'data/Rupes A and B'
OUTPUT_ROOT = 'outcome/Rupes A and B'

In [19]:
# Create a folder for the source data, if it does not exist
os.makedirs(OUTPUT_ROOT, exist_ok=True)

In [20]:
# Spectrogram parameters
fmin = 20
fmax = 1000
nperseg = 2456
nfft = 4096
noverlap = 1228
window = 'hann'

In [25]:
# Find all ".wav" and corresponding ".txt" file pairs
# Assume that filenames follow the structure: <base_name>.wav and <base_name>.Table.1.selections.txt
wav_files = glob.glob(os.path.join(DATA_ROOT, '*.wav'))

In [22]:

def compute_spectrogram(samples, sample_rate):
    """
    Computes the spectrogram for the entire signal.
    Returns: frequencies, times, spectrogram.
    """
    freqs, times, Sxx = signal.spectrogram(
        samples,
        sample_rate,
        nperseg=nperseg,
        nfft=nfft,
        noverlap=noverlap,
        window=window
    )
    # Remove very small values
    Sxx[Sxx < 0.001] = 0.001
    return freqs, times, Sxx

In [23]:
def freq_slice(freqs, Sxx, fmin, fmax):
    """
    Slices the spectrogram by frequencies [fmin, fmax].
    Returns the new freqs and Sxx.
    """
    idx = np.where((freqs >= fmin) & (freqs <= fmax))[0]
    freqs_new = freqs[idx]
    Sxx_new = Sxx[idx, :]
    return freqs_new, Sxx_new

In [24]:
def extract_call(freqs, times, Sxx, t_start, t_end, freq_range=(20, 1000)):
    """
    Extracts a segment of the spectrogram corresponding to [t_start, t_end] and the frequency range freq_range.
    Returns a spectrogram fragment (2D) and the corresponding axes (freqs_sub, times_sub).
    """
    # Time indices
    time_idx = np.where((times >= t_start) & (times <= t_end))[0]
    # Frequency indices
    freq_idx = np.where((freqs >= freq_range[0]) & (freqs <= freq_range[1]))[0]
    
    if len(time_idx) == 0 or len(freq_idx) == 0:
        return None, None, None  # In case the segment is empty

    Sxx_sub = Sxx[freq_idx][:, time_idx]
    freqs_sub = freqs[freq_idx]
    times_sub = times[time_idx]
    return Sxx_sub, freqs_sub, times_sub